# %% [markdown]
# # Step 1: Script Overview
# # -----------------------
# # This script automates running ESP-r simulations across ALL building folders.
# #
# # Key functionality:
# # - Iterates through all building directories in the baseline_models folder
# # - Identifies the model archetype folder inside each building
# # - Navigates to the cfg directory
# # - Verifies the presence of the required .cfg file
# # - Checks for required input .txt files
# # - Skips buildings with missing files (instead of crashing)
# # - Runs ESP-r `prj` in script mode for each valid input file
# # - Logs results and reports failures
# #
# # Designed for batch-processing large EPC datasets with robustness and traceability

In [8]:
# %% 
# Step 2: Import Required Libraries
# --------------------------------
import os
import subprocess
from pathlib import Path

In [9]:
# %%
# Step 3: Helper Function to Run Shell Commands
# --------------------------------------------
def run(cmd):
    print(f"\n>>> {cmd}")
    subprocess.run(cmd, shell=True, executable="/bin/bash")

In [10]:
# %%
# Step 4: Load ESP-r Environment
# -----------------------------
os.chdir(os.path.expanduser("~"))

run("source .profile")
run("which prj")  # confirm ESP-r is accessible


>>> source .profile

>>> which prj


In [11]:
# %%
# Step 5: Define Base Paths and Input Files
# ----------------------------------------
base_dir = Path("/Users/jakegehrung/Desktop/data_raw/baseline_models")

input_files = [
    "ACH.txt",
    "latitude.txt",
    "longitude.txt",
    "roof.txt",
    "walls.txt",
    "windows.txt",
    "rotate.txt"
]

print(f"Base directory set to: {base_dir}")

Base directory set to: /Users/jakegehrung/Desktop/data_raw/baseline_models


In [12]:
# %%
# Step 6: Identify All Building Directories
# ----------------------------------------
building_dirs = [d for d in base_dir.iterdir() if d.is_dir()]

print(f"Found {len(building_dirs)} building folders.")

Found 117 building folders.


In [ ]:
# %%
# Step 7 (UPDATED): Robust Detection of Archetype and CFG Folder
# --------------------------------------------------------------
valid_archetypes = {
    "SemiDetached_2F",
    "Detached_2F"
}

summary_results = {}
skipped_buildings = []

for building in building_dirs:
    print(f"\n========== Processing Building: {building.name} ==========")
    
    # Find valid archetype folder by name match
    archetype_dir = None
    for sub in building.iterdir():
        if sub.is_dir() and sub.name in valid_archetypes:
            archetype_dir = sub
            break
    
    if archetype_dir is None:
        print("No valid archetype folder found. Skipping.")
        skipped_buildings.append(building.name)
        continue
    
    print(f"Found archetype folder: {archetype_dir.name}")
    
    cfg_dir = archetype_dir / "cfg"
    
    if not cfg_dir.exists():
        print("No cfg folder found inside archetype. Skipping.")
        skipped_buildings.append(building.name)
        continue
    
    print(f"Found cfg directory: {cfg_dir}")
    
    cfg_file = cfg_dir / "SemiDetached_2F.cfg"
    
    if not cfg_file.exists():
        print("CFG file missing. Skipping.")
        skipped_buildings.append(building.name)
        continue
    
    # Check input files
    missing_files = [f for f in input_files if not (cfg_dir / f).exists()]
    
    if missing_files:
        print(f"Missing input files: {missing_files}. Skipping building.")
        skipped_buildings.append(building.name)
        continue
    
    print("All required files found. Running simulations...")
    
    os.chdir(cfg_dir)
    
    building_results = {}
    
    for input_file in input_files:
        print(f"\n--- Running: {input_file} ---")
        
        cmd = f"""
        source ~/.profile
        cd "{cfg_dir}"
        prj -mode script -file SemiDetached_2F.cfg < {input_file}
        """
        
        result = subprocess.run(
            cmd,
            shell=True,
            executable="/bin/bash",
            capture_output=True,
            text=True
        )
        
        building_results[input_file] = result
        
        print("Return Code:", result.returncode)
        print("\nSTDOUT:\n", result.stdout)
        print("\nSTDERR:\n", result.stderr)
    
    summary_results[building.name] = building_results


========== Processing Building: 1001664924 ==========
Found archetype folder: SemiDetached_2F
Found cfg directory: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001664924/SemiDetached_2F/cfg
Missing input files: ['ACH.txt', 'latitude.txt', 'longitude.txt', 'roof.txt', 'walls.txt', 'windows.txt', 'rotate.txt']. Skipping building.

========== Processing Building: 1001829085 ==========
Found archetype folder: SemiDetached_2F
Found cfg directory: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001829085/SemiDetached_2F/cfg
Missing input files: ['ACH.txt', 'latitude.txt', 'longitude.txt', 'roof.txt', 'walls.txt', 'windows.txt', 'rotate.txt']. Skipping building.

========== Processing Building: 1001063639 ==========
Found archetype folder: Detached_2F_2
Found cfg directory: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001063639/Detached_2F_2/cfg
Missing input files: ['ACH.txt', 'latitude.txt', 'longitude.txt', 'roof.txt', 'walls.txt', 'windows.txt', 'rotate.txt']. Skippi

In [14]:
# %%
# Step 8: Summary of Results
# -------------------------
failed_runs = []

for building, results in summary_results.items():
    for file, result in results.items():
        if result.returncode != 0:
            failed_runs.append((building, file))

print("\n========== SUMMARY ==========")

if failed_runs:
    print("\nFailed simulations:")
    for b, f in failed_runs:
        print(f"- Building {b}, File: {f}")
else:
    print("\nAll simulations completed successfully.")

if skipped_buildings:
    print("\nSkipped buildings:")
    for b in skipped_buildings:
        print(f"- {b}")
else:
    print("\nNo buildings were skipped.")


========== SUMMARY ==========

All simulations completed successfully.

Skipped buildings:
- 1001664924
- 1001829085
- 1001063639
- 1001829071
- 1234761002
- 1002539407
- 1001063637
- 1001664941
- 1001991633
- 1235057414
- 1001829079
- 1001664922
- 1234761003
- 1001664925
- 1000238907
- 1234761004
- 1002313096
- 1001870878
- 1001664940
- 1234982990
- 1234806523
- 1001870876
- 1001870882
- 1002143357
- 1001951902
- 1234621926
- 1234647955
- 1001906269
- 1001951903
- 1001627539
- 1002143351
- 1001627541
- 1236594950
- 1001627570
- 1002031280
- 1001627549
- 1001627540
- 1001627547
- 1001870854
- 1001430744
- 1234760995
- 1235812262
- 1000336709
- 1001991628
- 1001664938
- 1000238925
- 1002143291
- 1000238914
- 1001829063
- 1001951854
- 1001870864
- 1001991629
- 1001951898
- 1002473722
- 1001063646
- 1001991627
- 1001870855
- 1236034494
- 1001664930
- 1001664937
- 1001829065
- 1000238923
- 1001664939
- 1000238924
- 1234806526
- 1001664942
- 1000336711
- 1002313093
- 1234761001
- 100195188